In [1]:
import torch
import numpy
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
import imageio.v2 as imageio

from src.utils.visualization import *
from src.utils.visualization import _add_aligned_colorbar
from PIL import Image
from IPython.display import HTML

from src.datasets.vitae_dataset import load_data as load_vitae
from src.datasets.voronoi_datasets import load_data as load_voronoi
from src.datasets.utils import read_real_observation_files

save_dir = "presentation_images"
os.makedirs(save_dir, exist_ok=True)

In [2]:
d_polair_o3 = np.load('data/d_polair_O3.npy')
d_polair_pm10 = np.load('data/d_polair_PM10.npy')
d_polair_pm25 = np.load('data/d_polair_PM25.npy')
d_polair_no2 = np.load('data/d_polair_NO2.npy')

all_pollutants = np.concatenate([
    d_polair_o3,
    d_polair_pm10,
    d_polair_pm25,
    d_polair_no2
], axis=1)

train_vit, _, test_vit, _ = load_vitae(combine_train_val=True, scaling_type='none')

all_pollutants_observations = np.concatenate(
    [[vitae_batch[0] for vitae_batch in train_vit],
    [vitae_batch[0] for vitae_batch in test_vit]]
)

train_vor, _, test_vor, _ = load_voronoi(combine_train_val=True, scaling_type='none')

all_pollutants_voronoi = np.concatenate(
    [[vitae_batch[0] for vitae_batch in train_vit],
    [vitae_batch[0] for vitae_batch in test_vit]]
)

real_data = read_real_observation_files()

pollutant_names = ['O3', 'PM10', 'PM2.5', 'NO2']

print("All pollutants shape:", all_pollutants.shape)
print("All pollutants observations shape:", all_pollutants_observations.shape)
print("All pollutants voronoi shape:", all_pollutants_voronoi.shape)
print("Real data shape:", real_data.shape)

All pollutants shape: (8518, 4, 75, 110)
All pollutants observations shape: (8518, 4, 75, 110)
All pollutants voronoi shape: (8518, 4, 75, 110)
Real data shape: (8759, 4, 75, 110)


In [ ]:
for pol_idx, pol_name in enumerate(pollutant_names):
    plt.figure(figsize=(7, 10))
    plt.imshow(all_pollutants[100, pol_idx], cmap='viridis')
    plt.axis('off')
    plt.savefig(
        f"{save_dir}/example_{pol_name}.png", 
        dpi=300, 
        bbox_inches="tight", 
        pad_inches=0, 
    )

In [ ]:
matplotlib.use("Agg")  # headless-safe

N = 200  # number of frames to include

for pol_idx, pol_name in enumerate(pollutant_names):
    frames = []
    for t in range(N):
        img = all_pollutants[t, pol_idx]

        # Compute figure size exactly matching image shape
        h, w = img.shape
        fig = plt.figure(
            figsize=(7, 10),
            dpi=300,
            frameon=False,
            facecolor=None
        )

        ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(img, cmap="viridis")
        ax.axis("off")

        # Draw and capture
        fig.canvas.draw()
        frame = np.asarray(fig.canvas.buffer_rgba())
        frames.append(frame)

        plt.close(fig)

    gif_path = f"{save_dir}/gif_{pol_name}.gif"
    imageio.mimsave(gif_path, frames, fps=10, loop=0)
    print(f"Saved {gif_path}")


In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(18, 8))

v_max = [
    np.max(all_pollutants[[1, 100, 5000], 0]),
    np.max(all_pollutants[[1, 100, 5000], 1]),
    np.max(all_pollutants[[1, 100, 5000], 2]),
    np.max(all_pollutants[[1, 100, 5000], 3])
]

dates = [
    "1st January",
    "5th January",
    "28 July"
]
for idx, t in enumerate([1, 100, 5000]):

    im = axs[idx][0].imshow(all_pollutants_observations[t][0], vmin=0, vmax=v_max[0])
    divider = make_axes_locatable(axs[idx][0])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][1].imshow(all_pollutants_observations[t][1], vmin=0, vmax=v_max[1])
    divider = make_axes_locatable(axs[idx][1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][2].imshow(all_pollutants_observations[t][2], vmin=0, vmax=v_max[2])
    divider = make_axes_locatable(axs[idx][2])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][3].imshow(all_pollutants_observations[t][3], vmin=0, vmax=v_max[3])
    divider = make_axes_locatable(axs[idx][3])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    if idx == 0:
        axs[idx][0].set_title('O3', fontsize=16)
        axs[idx][1].set_title('PM10', fontsize=16)
        axs[idx][2].set_title('PM2.5', fontsize=16)
        axs[idx][3].set_title('NO2', fontsize=16)

    axs[idx][0].set_ylabel(f'{dates[idx]}', fontsize=16)

for ax in axs.flatten():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(f"{save_dir}/example_observations.png", dpi=300, bbox_inches="tight")
plt.close()

fig, axs = plt.subplots(3, 4, figsize=(18, 8))

for idx, t in enumerate([1, 100, 5000]):

    im = axs[idx][0].imshow(all_pollutants[t][0], vmin=0, vmax=v_max[0])
    divider = make_axes_locatable(axs[idx][0])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][1].imshow(all_pollutants[t][1], vmin=0, vmax=v_max[1])
    divider = make_axes_locatable(axs[idx][1])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][2].imshow(all_pollutants[t][2], vmin=0, vmax=v_max[2])
    divider = make_axes_locatable(axs[idx][2])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    axs[idx][3].imshow(all_pollutants[t][3], vmin=0, vmax=v_max[3])
    divider = make_axes_locatable(axs[idx][3])
    cax = divider.append_axes("right", size="5%", pad=0.05)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=12)

    if idx == 0:
        axs[idx][0].set_title('O3', fontsize=16)
        axs[idx][1].set_title('PM10', fontsize=16)
        axs[idx][2].set_title('PM2.5', fontsize=16)
        axs[idx][3].set_title('NO2', fontsize=16)

    axs[idx][0].set_ylabel(f'{dates[idx]}', fontsize=16)
    ax.set_xticks([])
    ax.set_yticks([])

for ax in axs.flatten():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig(f"{save_dir}/example_full.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
def animate_dataset(dataset: np.ndarray, num_frames: int, save_dir: str=None, filename: str="animation.gif"):
    v_maxs = [min(np.nanmax(dataset[:num_frames, i]), 250) for i in range(len(pollutant_names))]

    fig, axs = plt.subplots(2, 2, figsize=(11, 8))
    axs = axs.flatten()

    pred_images = []

    for i, pollutant in enumerate(pollutant_names):
        im_pred = axs[i].imshow(dataset[0, i], animated=True, cmap='viridis', vmin=0, vmax=v_maxs[i])
        axs[i].set_title(f"{pollutant} Observations")
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        pred_images.append(im_pred)
        _add_aligned_colorbar(im_pred, axs[i])

    def update(frame):
        for i in range(len(pollutant_names)):
            pred_images[i].set_array(dataset[frame, i])
        return pred_images

    ani = animation.FuncAnimation(fig, update, frames=num_frames, blit=True, interval=200)
    plt.tight_layout()
    plt.close(fig)

    # Save as GIF or return HTML
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, filename)
        ani.save(save_path, writer='pillow', fps=5)

        reset_theme()
        return None
    else:
        reset_theme()
        return HTML(ani.to_jshtml())
    
animate_dataset(all_pollutants, num_frames=200, save_dir=save_dir, filename="synthethic_animation.gif")
animate_dataset(real_data, num_frames=200, save_dir=save_dir, filename="real_animation.gif")

In [3]:
animate_predictions(
    np.load('results/predictions/clstm/sparse_fine_tuned_no_pre_42_predictions.npz')["predictions"],
    num_frames=200,
    save_dir=save_dir,
    filename="clstm_joint.gif"
)

In [20]:
def animate_sensor_placement(predictions: np.ndarray, num_frames: int, save_dir: str=None, filename: str="predictions_animation.gif"):
    sns.set_theme(style="white", context="talk", font_scale=1)
    
    pollutants = ["O3", "PM10", "PM25", "NO2"]

    fig, axs = plt.subplots(1, 4, figsize=(22, 4))

    pred_images = []

    for i, pollutant in enumerate(pollutants):
        im_pred = axs[i].imshow(predictions[0, i] != 0, animated=True, cmap='viridis', vmin=0.0, vmax=1.0)
        axs[i].set_title(f"{pollutant} Sensors")
        axs[i].set_xticks([])
        axs[i].set_yticks([])
        pred_images.append(im_pred)

    def update(frame):
        for i in range(len(pollutants)):
            pred_images[i].set_array(predictions[frame, i] > 1)
        return pred_images

    ani = animation.FuncAnimation(fig, update, frames=num_frames, blit=True, interval=200)
    plt.tight_layout()
    plt.close(fig)

    # Save as GIF or return HTML
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, filename)
        ani.save(save_path, writer='pillow', fps=5)

        reset_theme()
        return None
    else:
        reset_theme()
        return HTML(ani.to_jshtml())

animate_sensor_placement(
    real_data.sum(axis=0)[np.newaxis, ...],
    num_frames=1,
    save_dir=save_dir,
    filename="sensors_real.gif"
)